# Modelling

This focuses on developing and evaluating predictive models to estimate the likelihood of loan default. Multiple machine learning algorithms are trained and compared using a consistent evaluation framework. 

Given the business objective of minimizing missed defaults, particular emphasis is placed on recall while also considering precision, F1-score, and ROC-AUC. The modelling process includes baseline model development, hyperparameter tuning, threshold optimization, and performance comparison to identify the most effective model for credit risk prediction.

### Import and load

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import optuna
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from scipy.stats import ks_2samp
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation


X_train = pd.read_parquet('../data/processed/X_train.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet').squeeze()
X_val   = pd.read_parquet('../data/processed/X_val.parquet')
y_val   = pd.read_parquet('../data/processed/y_val.parquet').squeeze()

print(f'Train: {len(X_train):,} rows | Val: {len(X_val):,} rows')

Train: 61,442 rows | Val: 21,786 rows


In [2]:
# Summary stat of X-train
print(X_train.describe().T[['mean', 'min', 'max']])

                           mean          min          max
loan_to_income         0.217900     0.005000      3.80000
dti                   18.002290     0.000000     38.33000
fico_orig            696.967302   627.000000    847.50000
revol_util            54.871402     0.000000    193.00000
employment_years       5.708392     0.000000     10.00000
short_tenure_flag      0.131457     0.000000      1.00000
vintage_year        2013.985596  2007.000000   2015.00000
vintage_quarter        2.694948     1.000000      4.00000
open_acc_band          1.630302     0.000000      3.00000
delinq_flag            0.192311     0.000000      1.00000
pub_rec_flag           0.155545     0.000000      1.00000
purpose_risk_tier      0.978012     0.000000      2.00000
loan_amnt          14363.257218  1000.000000  35000.00000
log_annual_inc        11.062272     8.006701     12.42922
home_ownership         4.080027     0.000000      5.00000


In [3]:
# Class Imbalance Weight Calculation
# To account for class imbalance during model training, the `scale_pos_weight` parameter 
# was calculated as the ratio of non-default loans to default loans in the training dataset.

scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

scale_pos_weight: 4.43


A `scale_pos_weight` of **4.43** implies that, on average, there is approximately **1 defaulted loan for every 5 loans in the training data**, with the remaining 4 loans being non-defaults. This closely aligns with the class distribution observed during the EDA phase, indicating that the train-test split successfully preserved the underlying structure of the dataset. Applying this weight helps the model pay greater attention to the less frequent default cases, reducing the risk of overlooking potentially high-risk borrowers and supporting the project's objective of maximizing default detection.

## Model 1 — Logistic Regression

Because the training data contains approximately **1 defaulted loan for every 5 non-default loans**, `class_weight='balanced'` is used to prevent Logistic Regression from being biased toward the more common non-default class. This helps the model pay greater attention to default cases and improves its ability to identify high-risk borrowers without altering the underlying data.

Logistic Regression was trained on standardized features because its optimization process is sensitive to differences in feature scale. In contrast, XGBoost was trained on the original feature values because tree-based algorithms make decisions using feature thresholds rather than feature magnitudes, making them inherently insensitive to scaling.

In [4]:
# Scale features since Logistic Regression is sensitive to feature magnitude
scaler = StandardScaler()
X_train_lr = scaler.fit_transform(X_train)
X_val_lr   = scaler.transform(X_val)

# Train Logistic Regression with class balancing to address the ~1:5 default/non-default distribution
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_lr, y_train)

# Evaluate discriminatory power using ROC-AUC
lr_auc = roc_auc_score(y_val, lr_model.predict_proba(X_val_lr)[:,1])
print(f'Logistic Regression Val AUC: {lr_auc:.4f}')

Logistic Regression Val AUC: 0.6607


**Findings:**

The Logistic Regression model achieved a validation ROC-AUC of **0.6607**. This means that when comparing a randomly selected defaulted loan and a randomly selected non-defaulted loan, the model has approximately **66.1% probability of assigning a higher risk score to the defaulted loan**.

An ROC-AUC of 0.6526 indicates that the model performs better than random guessing (0.50) and captures meaningful patterns associated with default risk, although its discriminatory power remains modest. As a baseline model, this result provides a useful benchmark against which more advanced models can be evaluated.

In [5]:
# Save the trained Logistic Regression model and scaler for reuse during evaluation, explainability, and deployment
joblib.dump(lr_model, '../models/logistic_model.pkl')
joblib.dump(scaler,   '../models/lr_scaler.pkl')

print('Saved: models/logistic_model.pkl + models/lr_scaler.pkl')

Saved: models/logistic_model.pkl + models/lr_scaler.pkl


## Model 2 — XGBoost Objective Function with time-boxed 

#### Why XGBoost with Time-Boxed Optuna?

Rather than training XGBoost using manually selected hyperparameters, Optuna was used to automatically search for the combination of hyperparameters that maximized validation ROC-AUC. This data-driven approach reduces reliance on trial-and-error parameter selection and increases the likelihood of identifying a higher-performing model.

A time-boxed optimization strategy was chosen instead of a fixed number of trials, allowing Optuna to explore as many parameter combinations as possible within a predefined computational budget. This approach provides a practical balance between model performance and runtime efficiency while ensuring that tuning remains reproducible and scalable across different computing environments.

In [6]:
# Define Optuna objective function for tuning XGBoost hyperparameters
def xgb_objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1, 10, log=True),
        # High ceiling; early stopping determines the effective number of trees
        'n_estimators':          2000,
        'early_stopping_rounds': 50,
        'objective':        'binary:logistic',
        'eval_metric':      'auc',
        'scale_pos_weight': scale_pos_weight,
        'random_state':     42,
        'verbosity':        0,
        'n_jobs':           -1,
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

# Run Optuna with a seeded sampler (reproducible) and a 30-minute budget
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_xgb.optimize(xgb_objective, timeout=1800)

print(f'Best validation AUC (optimistic — tuning signal only): {study_xgb.best_value:.4f}')
print(f'Best params: {study_xgb.best_params}')

# Train final model using Optuna-selected hyperparameters
# Refit champion with early stopping retained (sets the correct tree count)
final_xgb = XGBClassifier(
    **study_xgb.best_params,
    n_estimators=2000,
    early_stopping_rounds=50,
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
)
final_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)


# Evaluate the final tuned model
# X_test is reserved and evaluated only once — on the final champion model to give an unbiased production estimate.
xgb_val_auc = roc_auc_score(y_val, final_xgb.predict_proba(X_val)[:, 1])
print(f'\nEffective trees (best_iteration_): {final_xgb.best_iteration}')
print(f'XGBoost Val AUC: {xgb_val_auc:.4f}')

Best validation AUC (optimistic — tuning signal only): 0.6694
Best params: {'max_depth': 4, 'learning_rate': 0.054389111244474764, 'subsample': 0.600409017095921, 'colsample_bytree': 0.6979630045408936, 'min_child_weight': 6, 'reg_lambda': 2.9569829910098493}

Effective trees (best_iteration_): 203
XGBoost Val AUC: 0.6694


## Model 3 — LightGBM with time-boxed Optuna

#### Why LightGBM?

LightGBM was selected as an additional gradient boosting algorithm to benchmark against XGBoost and other baseline models. Its leaf-wise tree growth strategy and computational efficiency enable it to capture complex non-linear relationships in borrower characteristics while often requiring less training time than traditional boosting approaches.

Hyperparameter optimization was performed to identify the LightGBM configuration that achieved the strongest validation ROC-AUC, ensuring that model performance was driven by empirical evidence rather than default parameter settings.

In [7]:
# Optuna objective for tuning LightGBM
def lgbm_objective(trial):
    params = {
        'num_leaves':        trial.suggest_int('num_leaves', 15, 127),
        'max_depth':         -1,
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'n_estimators':      2000,
        'objective':         'binary',
        'metric':            'auc',
        'scale_pos_weight':  scale_pos_weight,
        'random_state':      42,
        'verbosity':         -1,
        'n_jobs':            -1,
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[early_stopping(50, verbose=False), log_evaluation(0)])
    return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

# Run Optuna — seeded, quiet, 30-minute budget
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lgbm = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(lgbm_objective, timeout=1800)
print(f'Best validation AUC (tuning signal only): {study_lgbm.best_value:.4f}')
print(f'Best params: {study_lgbm.best_params}')

# Refit champion — keep early stopping, re-include fixed params
final_lgbm = LGBMClassifier(
    **study_lgbm.best_params,
    n_estimators=2000,
    max_depth=-1,
    subsample_freq=1,
    objective='binary',
    metric='auc',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbosity=-1,
    n_jobs=-1,
)
final_lgbm.fit(X_train, y_train,
               eval_set=[(X_val, y_val)],
               callbacks=[early_stopping(50, verbose=False), log_evaluation(0)])

# Validation AUC — model-selection metric; X_test held for NB10
lgbm_val_auc = roc_auc_score(y_val, final_lgbm.predict_proba(X_val)[:, 1])
print(f'\nEffective trees (best_iteration_): {final_lgbm.best_iteration_}')
print(f'LightGBM Val AUC: {lgbm_val_auc:.4f}')

Best validation AUC (tuning signal only): 0.6692
Best params: {'num_leaves': 23, 'min_child_samples': 168, 'learning_rate': 0.012884388206798783, 'subsample': 0.593500791586741, 'colsample_bytree': 0.7195294520547624, 'reg_lambda': 3.2394594536465746}

Effective trees (best_iteration_): 527
LightGBM Val AUC: 0.6692


In [8]:
# Save the trained XGBoost model and LightGBM model for reuse during evaluation, explainability, and deployment
joblib.dump(final_xgb, '../models/xgboost_model.pkl')
joblib.dump(final_lgbm,   '../models/lightgbm_model.pkl')

print('Saved: models/xgboost_model.pkl + models/lgbm.pkl')

Saved: models/xgboost_model.pkl + models/lgbm.pkl


## Evaluation of all 3 models on validation set

#### Validation, Champion Selection, Stability Assessment & Production Retrain

This phase evaluates the candidate models using a comprehensive set of credit-risk performance and stability metrics, including ROC-AUC (AUC), KS Statistic (KS), Gini Coefficient (Gini), and Population Stability Index (PSI). The objective is to identify the champion model based on predictive performance, discrimination power, calibration, and robustness.

Following model comparison and champion selection, the chosen model is retrained on the full 500,000-loan production dataset to maximize the use of available information before deployment. Additional temporal stability analysis is performed across loan vintages to assess whether model performance remains consistent over time and to identify potential drift in borrower behavior or portfolio characteristics.

The outputs from this stage represent the final model performance and stability results that are reported in the project report, executive summary, and professional portfolio.

In [15]:
# Quick comparison of models using AUC
print('\n=== Validation ROC-AUC Comparison ===')
print()
print(f'Logistic Regression: {lr_auc:.4f}')
print(f'LightGBM: {lgbm_val_auc:.4f}')
print(f'XGBoost: {xgb_val_auc:.4f}')


=== Validation ROC-AUC Comparison ===

Logistic Regression: 0.6607
LightGBM: 0.6692
XGBoost: 0.6694


In [16]:
# Metrics function to evaluate all 4 metrics of auc, ks, gini and brier
def compute_metrics(y_true, y_proba):
    auc   = roc_auc_score(y_true, y_proba)
    gini  = 2*auc - 1
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    ks    = max(tpr - fpr)
    brier = brier_score_loss(y_true, y_proba)
    return {'auc': round(auc,4), 'ks': round(ks,4),
            'gini': round(gini,4), 'brier': round(brier,4)}

print('Metrics defined.')

Metrics defined.


In [17]:
# Earlier saved models
models = {
    'Logistic': joblib.load('../models/logistic_model.pkl'),
    'XGBoost':  joblib.load('../models/xgboost_model.pkl'),
    'LightGBM': joblib.load('../models/lightgbm_model.pkl'),
}

print('=== Validation Set Metrics ===')
print()

val_results = {}
for name, model in models.items():
    X = scaler.transform(X_val) if name == 'Logistic' else X_val
    proba = model.predict_proba(X)[:,1]
    metrics = compute_metrics(y_val, proba)
    val_results[name] = metrics
    print(f'{name:20s}: AUC={metrics["auc"]:.4f}  KS={metrics["ks"]:.4f}  '
          f'Gini={metrics["gini"]:.4f}  Brier={metrics["brier"]:.4f}')

=== Validation Set Metrics ===

Logistic            : AUC=0.6607  KS=0.2343  Gini=0.3214  Brier=0.2362
XGBoost             : AUC=0.6694  KS=0.2488  Gini=0.3389  Brier=0.2321
LightGBM            : AUC=0.6692  KS=0.2442  Gini=0.3384  Brier=0.2313


In [21]:
for name, model in models.items():
    scores = cross_val_score(
        model,
        X = scaler.transform(X_val) if name == 'Logistic' else X_val,
        y_val,
        cv=3,
        scoring='roc_auc'
    )
    
    print(f"{name} ROC-AUC: {scores.mean():.3f}")

SyntaxError: positional argument follows keyword argument (2390074818.py, line 8)

In [23]:
# Feature Importance

importances = pd.Series(
    final_xgb.feature_importances_,
    index=X_train.columns[selector.get_support()]
).sort_values(ascending=False)

print(importances.head(15))

NameError: name 'selected_features' is not defined

In [20]:
# Champion = highest AUC
champion_name = max(val_results, key=lambda k: val_results[k]['auc'])
print(f'Champion: {champion_name}')
print()
print('>> Now retraining on 500k production sample...')
print('>> This is the ONLY time you use loans_prod_500k.parquet for training.')
print('>> Apply same feature engineering as NB08 to the production sample.')
print()
print('TODO: Copy the feature engineering cells from NB08 here,')
print('      apply to loans_prod_500k.parquet, refit the encoder on prod train set,')
print('      retrain champion with its best Optuna params,')
print('      save as models/champion_production.pkl')
print()
print('Placeholder — run after completing feature engineering on production sample:')
# joblib.dump(champion_prod, 'models/champion_production.pkl')

Champion: XGBoost

>> Now retraining on 500k production sample...
>> This is the ONLY time you use loans_prod_500k.parquet for training.
>> Apply same feature engineering as NB08 to the production sample.

TODO: Copy the feature engineering cells from NB08 here,
      apply to loans_prod_500k.parquet, refit the encoder on prod train set,
      retrain champion with its best Optuna params,
      save as models/champion_production.pkl

Placeholder — run after completing feature engineering on production sample:


In [19]:
print('=== Temporal Drift Analysis ===')
print('Evaluate model performance separately for each test vintage year.')
print()
print('TODO: Load production test set, compute AUC and KS for each of 2016, 2017, 2018.')
print()
# Example structure:
# drift = []
# for year in [2016, 2017, 2018]:
#     mask = (df_prod_test['vintage_year'] == year)
#     proba = champion_prod.predict_proba(X_prod_test_enc[mask])[:,1]
#     metrics = compute_metrics(y_prod_test[mask], proba)
#     drift.append({'vintage': year, **metrics})
#     print(f'{year}: AUC={metrics["auc"]:.4f}  KS={metrics["ks"]:.4f}')
print('Drift chart saved as report/figures/temporal_drift.png')

=== Temporal Drift Analysis ===
Evaluate model performance separately for each test vintage year.

TODO: Load production test set, compute AUC and KS for each of 2016, 2017, 2018.

Drift chart saved as report/figures/temporal_drift.png


## Model Comparison Findings

Because `grade`, `sub_grade`, and `int_rate` encode LendingClub's internal risk pricing, they were deliberately excluded from the feature set so the models would learn default risk directly from borrower and loan characteristics rather than replicate the lender's existing credit judgment. Under this leakage-free framework, three model families were evaluated on the held-out validation set: Logistic Regression (ROC-AUC = 0.6607), XGBoost (0.6693), and LightGBM (0.6692).

The two gradient boosting models delivered nearly identical performance and improved only marginally over the linear baseline, suggesting that predictive power is constrained primarily by the information available at loan origination rather than by model sophistication. Their close agreement indicates a stable performance ceiling for the selected feature set.

Given its slightly stronger validation ROC-AUC, XGBoost was chosen as the leading candidate and advanced to the final validation and production-readiness assessment stage.